# 遷移確率シミュレーション：評価ブレ混合条件下での微細ステップモデル
## Transition Probability Simulation with Evaluator Noise

- **等間隔遷移 / 加速劣化** の2シナリオ比較
- **頻度主義（カイ二乗検定）vs ベイズ的（ベイズ因子）** の同一集団捕捉率
- **N = 100 ~ 20,000** の大規模モンテカルロ・シミュレーション
- 結果: CSV + 7種のグラフ

Based on: Maeda (2020) 遷移確率の影響度を考慮した予防保全検討

In [ ]:
# -*- coding: utf-8 -*-
"""
=============================================================================
遷移確率シミュレーション：評価ブレ混合条件下での微細ステップモデル
Transition Probability Simulation with Evaluator Noise
=============================================================================

Google Colab用コード
- 等間隔遷移シナリオ / 加速劣化シナリオ
- 頻度主義（カイ二乗検定）vs ベイズ的（ベイズ因子）比較
- N = 100 ~ 20,000 の大規模シミュレーション
- CSV出力 + 可視化

Author: Based on Maeda (2020) framework
"""

# ============================================================================
# セル1: ライブラリのインストールとインポート
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
from scipy import stats
from scipy.special import gammaln
import warnings
import time
import os
from collections import defaultdict

warnings.filterwarnings('ignore')

# 日本語フォント設定（Google Colab対応）
try:
    # Google Colab環境
    import subprocess
    subprocess.run(['apt-get', '-y', 'install', 'fonts-noto-cjk'], 
                   capture_output=True, check=False)
    matplotlib.rcParams['font.family'] = 'Noto Sans CJK JP'
except:
    pass

matplotlib.rcParams['font.size'] = 12
matplotlib.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid')

print("=" * 70)
print("遷移確率シミュレーション：評価ブレ混合条件下での微細ステップモデル")
print("=" * 70)

# ============================================================================
# セル2: シミュレーションパラメータの定義
# ============================================================================

class SimulationConfig:
    """シミュレーション設定クラス"""
    
    # 真の遷移確率（年率）
    P_AB_TRUE = 0.10    # A→B 遷移確率
    P_BC_TRUE = 0.20    # B→C 遷移確率
    
    # 評価ノイズ
    SIGMA = 0.05         # 評価ブレの標準偏差
    
    # 微細時間ステップ
    N_STEPS = 12         # 観測周期内の分割数（月単位相当）
    
    # シミュレーション試行回数リスト
    N_SAMPLES_LIST = [100, 500, 1000, 2000, 5000, 10000, 20000]
    
    # 各Nにおけるシミュレーション反復回数（p値分布生成用）
    N_REPEATS = 500      # 各条件での反復回数
    
    # ベイズ事前分布のハイパーパラメータ（集中度）
    KAPPA_PRIOR = 100.0  # ディリクレ分布の集中度
    
    # 統計的有意水準
    ALPHA = 0.05
    
    # 出力ディレクトリ
    OUTPUT_DIR = "simulation_results"

config = SimulationConfig()
os.makedirs(config.OUTPUT_DIR, exist_ok=True)

print(f"\n【パラメータ設定】")
print(f"  真の遷移確率: P_AB={config.P_AB_TRUE}, P_BC={config.P_BC_TRUE}")
print(f"  評価ノイズ σ = {config.SIGMA}")
print(f"  微細ステップ数 n = {config.N_STEPS}")
print(f"  試行回数 N = {config.N_SAMPLES_LIST}")
print(f"  各条件での反復回数 = {config.N_REPEATS}")

# ============================================================================
# セル3: 微細ステップ遷移シミュレーション・エンジン
# ============================================================================

def compute_micro_step_probabilities(p_ab_annual, p_bc_annual, n_steps):
    """年率遷移確率を微細ステップ確率に変換"""
    p_ab_step = 1 - (1 - p_ab_annual) ** (1.0 / n_steps)
    p_bc_step = 1 - (1 - p_bc_annual) ** (1.0 / n_steps)
    return p_ab_step, p_bc_step


def simulate_single_unit_equal(p_ab_step, p_bc_step, n_steps, rng):
    """
    等間隔遷移シナリオ：1ユニットの1年間の遷移をシミュレート
    各ステップで一定の遷移確率
    """
    state = 0  # 0=A, 1=B, 2=C
    for _ in range(n_steps):
        if state == 0:  # A
            if rng.random() < p_ab_step:
                state = 1  # A→B
        elif state == 1:  # B
            if rng.random() < p_bc_step:
                state = 2  # B→C
        # state == 2 (C) は吸収状態
    return state


def simulate_single_unit_accel(p_ab_step, p_bc_base_step, n_steps, rng,
                                accel_factor=1.5):
    """
    加速劣化シナリオ：状態Bに入った後、時間経過とともに劣化加速
    p_bc が指数関数的に増大する
    """
    state = 0
    steps_in_b = 0
    for step_idx in range(n_steps):
        if state == 0:
            if rng.random() < p_ab_step:
                state = 1
                steps_in_b = 0
        elif state == 1:
            # 加速劣化：Bに滞在するほど遷移確率が上昇
            accel = accel_factor ** (steps_in_b / n_steps)
            p_bc_current = min(p_bc_base_step * accel, 0.95)
            if rng.random() < p_bc_current:
                state = 2
            else:
                steps_in_b += 1
    return state


def simulate_population(n_units, scenario, p_ab_annual, p_bc_annual,
                         n_steps, sigma, rng):
    """
    N個のユニット集団をシミュレートし、評価ノイズを付加
    
    Returns:
        observed_counts: [n_A, n_B, n_C] 観測された状態カウント
        true_counts: [n_A, n_B, n_C] 真の状態カウント（ノイズなし）
    """
    p_ab_step, p_bc_step = compute_micro_step_probabilities(
        p_ab_annual, p_bc_annual, n_steps
    )
    
    # 各ユニットの真の状態をシミュレート
    true_states = np.zeros(n_units, dtype=int)
    for i in range(n_units):
        if scenario == "equal":
            true_states[i] = simulate_single_unit_equal(
                p_ab_step, p_bc_step, n_steps, rng
            )
        elif scenario == "accel":
            true_states[i] = simulate_single_unit_accel(
                p_ab_step, p_bc_step, n_steps, rng
            )
    
    # 真の状態カウント
    true_counts = np.array([
        np.sum(true_states == 0),
        np.sum(true_states == 1),
        np.sum(true_states == 2)
    ], dtype=float)
    
    # 評価ノイズの付加（正規分布ノイズを構成比に加算）
    true_proportions = true_counts / n_units
    noise = rng.normal(0, sigma, size=3)
    observed_proportions = true_proportions + noise
    
    # 負値の補正と正規化
    observed_proportions = np.maximum(observed_proportions, 0.001)
    observed_proportions /= observed_proportions.sum()
    
    # カウントに変換（四捨五入で合計をN に調整）
    observed_counts = np.round(observed_proportions * n_units).astype(int)
    diff = n_units - observed_counts.sum()
    observed_counts[np.argmax(observed_counts)] += diff
    
    return observed_counts, true_counts.astype(int)


# ============================================================================
# セル4: 統計的検定関数（頻度主義 + ベイズ）
# ============================================================================

def compute_theoretical_distribution(p_ab_annual, p_bc_annual, n_steps,
                                      scenario):
    """理論的な状態分布を数値的に算出（大数シミュレーション）"""
    rng = np.random.default_rng(42)
    n_large = 100000
    states = np.zeros(n_large, dtype=int)
    
    p_ab_step, p_bc_step = compute_micro_step_probabilities(
        p_ab_annual, p_bc_annual, n_steps
    )
    
    for i in range(n_large):
        if scenario == "equal":
            states[i] = simulate_single_unit_equal(
                p_ab_step, p_bc_step, n_steps, rng
            )
        else:
            states[i] = simulate_single_unit_accel(
                p_ab_step, p_bc_step, n_steps, rng
            )
    
    theoretical = np.array([
        np.sum(states == 0) / n_large,
        np.sum(states == 1) / n_large,
        np.sum(states == 2) / n_large
    ])
    return theoretical


def chi_square_test(observed_counts, expected_proportions):
    """カイ二乗適合度検定"""
    n = observed_counts.sum()
    expected_counts = expected_proportions * n
    
    # 期待度数が0のセルを回避
    mask = expected_counts > 0
    if mask.sum() < 2:
        return 1.0, 0.0
    
    chi2, p_value = stats.chisquare(
        observed_counts[mask], f_exp=expected_counts[mask]
    )
    return p_value, chi2


def log_marginal_likelihood_dirichlet(counts, alpha_prior):
    """
    ディリクレ-多項分布の対数周辺尤度を計算
    log p(x|alpha) = log B(alpha + x) - log B(alpha)
    where B is the multivariate beta function
    """
    alpha_post = alpha_prior + counts
    
    log_ml = (
        gammaln(alpha_prior.sum()) - gammaln(alpha_post.sum())
        + np.sum(gammaln(alpha_post) - gammaln(alpha_prior))
    )
    return log_ml


def bayes_factor_test(observed_counts, theoretical_proportions, 
                       kappa=100.0, sigma=0.05):
    """
    ベイズ因子による同一集団判定
    
    H0: 観測データは理論モデルと同一集団（事前分布＝理論分布×集中度）
    H1: 観測データは一様分布（無情報事前分布）
    
    Returns:
        bf_01: H0を支持するベイズ因子（>1でH0支持）
        posterior_mean: 事後分布の平均
    """
    counts = observed_counts.astype(float)
    
    # H0の事前分布：理論分布に基づくディリクレ分布
    # σを考慮して集中度を調整
    effective_kappa = kappa * (1 - sigma * 2)  # σが大きいほど確信度を下げる
    alpha_h0 = theoretical_proportions * effective_kappa + 1.0
    
    # H1の事前分布：弱情報ディリクレ分布（ほぼ一様）
    alpha_h1 = np.ones(3) * 1.0
    
    # 周辺尤度の計算
    log_ml_h0 = log_marginal_likelihood_dirichlet(counts, alpha_h0)
    log_ml_h1 = log_marginal_likelihood_dirichlet(counts, alpha_h1)
    
    # ベイズ因子（対数スケール）
    log_bf_01 = log_ml_h0 - log_ml_h1
    bf_01 = np.exp(np.clip(log_bf_01, -500, 500))
    
    # 事後分布の平均
    alpha_post = alpha_h0 + counts
    posterior_mean = alpha_post / alpha_post.sum()
    
    return bf_01, posterior_mean


# ============================================================================
# セル5: メインシミュレーション実行エンジン
# ============================================================================

def run_full_simulation(config):
    """
    全シナリオ × 全N に対するシミュレーションを実行
    """
    scenarios = ["equal", "accel"]
    scenario_names = {"equal": "等間隔遷移", "accel": "加速劣化"}
    
    # 理論分布の事前計算
    print("\n[1/3] 理論分布の計算中...")
    theoretical_dists = {}
    for sc in scenarios:
        theoretical_dists[sc] = compute_theoretical_distribution(
            config.P_AB_TRUE, config.P_BC_TRUE, config.N_STEPS, sc
        )
        print(f"  {scenario_names[sc]}: P_A={theoretical_dists[sc][0]:.4f}, "
              f"P_B={theoretical_dists[sc][1]:.4f}, P_C={theoretical_dists[sc][2]:.4f}")
    
    # 結果格納用
    all_results = []
    p_value_distributions = defaultdict(list)
    bf_distributions = defaultdict(list)
    
    total_runs = len(scenarios) * len(config.N_SAMPLES_LIST) * config.N_REPEATS
    current_run = 0
    
    print(f"\n[2/3] メインシミュレーション実行中... (総計算回数: {total_runs:,})")
    start_time = time.time()
    
    for sc in scenarios:
        theo_dist = theoretical_dists[sc]
        
        for n_samples in config.N_SAMPLES_LIST:
            rng = np.random.default_rng(seed=42 + n_samples)
            
            p_values_chi2 = []
            chi2_stats_list = []
            bf_values = []
            capture_success_chi2 = 0
            capture_success_bayes = 0
            
            for rep in range(config.N_REPEATS):
                # シミュレーション実行
                obs_counts, true_counts = simulate_population(
                    n_samples, sc, config.P_AB_TRUE, config.P_BC_TRUE,
                    config.N_STEPS, config.SIGMA, rng
                )
                
                # カイ二乗検定
                p_val, chi2_stat = chi_square_test(obs_counts, theo_dist)
                p_values_chi2.append(p_val)
                chi2_stats_list.append(chi2_stat)
                
                if p_val > config.ALPHA:
                    capture_success_chi2 += 1
                
                # ベイズ因子検定
                bf, post_mean = bayes_factor_test(
                    obs_counts, theo_dist, config.KAPPA_PRIOR, config.SIGMA
                )
                bf_values.append(bf)
                
                # BF > 1/3 で同一集団と判定（Jeffreys基準）
                if bf > 1.0 / 3.0:
                    capture_success_bayes += 1
                
                current_run += 1
            
            # 結果集計
            capture_rate_chi2 = capture_success_chi2 / config.N_REPEATS * 100
            capture_rate_bayes = capture_success_bayes / config.N_REPEATS * 100
            
            result = {
                "scenario": sc,
                "scenario_name": scenario_names[sc],
                "N": n_samples,
                "p_value_mean": np.mean(p_values_chi2),
                "p_value_median": np.median(p_values_chi2),
                "p_value_std": np.std(p_values_chi2),
                "p_value_q25": np.percentile(p_values_chi2, 25),
                "p_value_q75": np.percentile(p_values_chi2, 75),
                "chi2_mean": np.mean(chi2_stats_list),
                "capture_rate_chi2": capture_rate_chi2,
                "bf_mean": np.mean(bf_values),
                "bf_median": np.median(bf_values),
                "log_bf_mean": np.mean(np.log10(np.maximum(bf_values, 1e-300))),
                "capture_rate_bayes": capture_rate_bayes,
            }
            all_results.append(result)
            
            # p値分布・BF分布の保存
            key = (sc, n_samples)
            p_value_distributions[key] = np.array(p_values_chi2)
            bf_distributions[key] = np.array(bf_values)
            
            elapsed = time.time() - start_time
            progress = current_run / total_runs * 100
            print(f"  [{progress:5.1f}%] {scenario_names[sc]} N={n_samples:>6,}: "
                  f"捕捉率(χ²)={capture_rate_chi2:.1f}%, "
                  f"捕捉率(Bayes)={capture_rate_bayes:.1f}% "
                  f"[{elapsed:.1f}s]")
    
    print(f"\n[3/3] シミュレーション完了! 総実行時間: {time.time()-start_time:.1f}秒")
    
    results_df = pd.DataFrame(all_results)
    return results_df, p_value_distributions, bf_distributions, theoretical_dists


# ============================================================================
# セル6: シミュレーション実行
# ============================================================================

print("\n" + "=" * 70)
print("シミュレーション開始")
print("=" * 70)

results_df, p_val_dists, bf_dists, theo_dists = run_full_simulation(config)

# ============================================================================
# セル7: CSV出力
# ============================================================================

# メイン結果CSV
csv_path_main = os.path.join(config.OUTPUT_DIR, "main_results.csv")
results_df.to_csv(csv_path_main, index=False, encoding='utf-8-sig')
print(f"\n結果CSV保存: {csv_path_main}")

# p値分布の詳細CSV
p_val_records = []
for (sc, n_samples), p_vals in p_val_dists.items():
    for i, pv in enumerate(p_vals):
        p_val_records.append({
            "scenario": sc,
            "N": n_samples,
            "repeat_id": i,
            "p_value": pv,
            "bf_value": bf_dists[(sc, n_samples)][i]
        })

p_val_df = pd.DataFrame(p_val_records)
csv_path_pval = os.path.join(config.OUTPUT_DIR, "p_value_distributions.csv")
p_val_df.to_csv(csv_path_pval, index=False, encoding='utf-8-sig')
print(f"p値分布CSV保存: {csv_path_pval}")

# 結果サマリー表示
print("\n" + "=" * 70)
print("【結果サマリー】")
print("=" * 70)
display_cols = ["scenario_name", "N", "p_value_mean", "p_value_median", 
                "capture_rate_chi2", "capture_rate_bayes"]
print(results_df[display_cols].to_string(index=False))

# ============================================================================
# セル8: 可視化1 - 捕捉成功率の比較（メインチャート）
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for idx, sc in enumerate(["equal", "accel"]):
    ax = axes[idx]
    df_sc = results_df[results_df["scenario"] == sc]
    
    x = np.arange(len(df_sc))
    width = 0.35
    
    bars1 = ax.bar(x - width/2, df_sc["capture_rate_chi2"], width,
                    label="Chi-square (frequentist)", color='#E74C3C', alpha=0.8)
    bars2 = ax.bar(x + width/2, df_sc["capture_rate_bayes"], width,
                    label="Bayesian (Bayes Factor)", color='#2E86C1', alpha=0.8)
    
    ax.set_xlabel("Trial Count N", fontsize=13)
    ax.set_ylabel("Capture Success Rate (%)", fontsize=13)
    title = "Equal-Interval" if sc == "equal" else "Accelerated Degradation"
    ax.set_title(f"Capture Rate: {title}\n"
                 f"(sigma={config.SIGMA}, P_AB={config.P_AB_TRUE}, P_BC={config.P_BC_TRUE})",
                 fontsize=13)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{n:,}" for n in df_sc["N"]], rotation=45)
    ax.set_ylim(0, 105)
    ax.axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95% target')
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    
    # バーの上に値を表示
    for bar in bars1:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1,
                f'{h:.0f}%', ha='center', va='bottom', fontsize=9)
    for bar in bars2:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., h + 1,
                f'{h:.0f}%', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
fig_path1 = os.path.join(config.OUTPUT_DIR, "capture_rate_comparison.png")
plt.savefig(fig_path1, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path1}")

# ============================================================================
# セル9: 可視化2 - p値分布のヒストグラム（N別・シナリオ別）
# ============================================================================

n_list_plot = [100, 1000, 5000, 20000]  # 代表的なN値

for sc in ["equal", "accel"]:
    fig, axes = plt.subplots(1, len(n_list_plot), figsize=(20, 5))
    sc_label = "Equal-Interval" if sc == "equal" else "Accelerated"
    
    for col_idx, n_samples in enumerate(n_list_plot):
        ax = axes[col_idx]
        key = (sc, n_samples)
        
        if key in p_val_dists:
            p_vals = p_val_dists[key]
            
            ax.hist(p_vals, bins=30, color='#3498DB', alpha=0.7, edgecolor='white',
                    density=True)
            ax.axvline(x=0.05, color='red', linestyle='--', linewidth=2,
                       label=f'alpha=0.05')
            
            # 捕捉率を表示
            rate = np.mean(p_vals > 0.05) * 100
            ax.text(0.95, 0.95, f'p>0.05: {rate:.1f}%',
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
        
        ax.set_xlabel("p-value", fontsize=11)
        ax.set_ylabel("Density" if col_idx == 0 else "", fontsize=11)
        ax.set_title(f"N = {n_samples:,}", fontsize=12)
        ax.set_xlim(-0.02, 1.02)
        ax.legend(fontsize=9)
    
    fig.suptitle(f"p-value Distribution ({sc_label}, sigma={config.SIGMA})",
                 fontsize=14, y=1.02)
    plt.tight_layout()
    fig_path = os.path.join(config.OUTPUT_DIR, f"p_value_histogram_{sc}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved: {fig_path}")

# ============================================================================
# セル10: 可視化3 - ベイズ因子の分布
# ============================================================================

for sc in ["equal", "accel"]:
    fig, axes = plt.subplots(1, len(n_list_plot), figsize=(20, 5))
    sc_label = "Equal-Interval" if sc == "equal" else "Accelerated"
    
    for col_idx, n_samples in enumerate(n_list_plot):
        ax = axes[col_idx]
        key = (sc, n_samples)
        
        if key in bf_dists:
            bf_vals = bf_dists[key]
            log_bf = np.log10(np.maximum(bf_vals, 1e-300))
            
            ax.hist(log_bf, bins=30, color='#27AE60', alpha=0.7, edgecolor='white',
                    density=True)
            # BF = 1/3 の閾値線
            ax.axvline(x=np.log10(1/3), color='red', linestyle='--', linewidth=2,
                       label='BF=1/3 threshold')
            ax.axvline(x=0, color='orange', linestyle=':', linewidth=1.5,
                       label='BF=1 (neutral)')
            
            rate = np.mean(bf_vals > 1/3) * 100
            ax.text(0.95, 0.95, f'BF>1/3: {rate:.1f}%',
                    transform=ax.transAxes, ha='right', va='top',
                    fontsize=11, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.8))
        
        ax.set_xlabel("log10(Bayes Factor)", fontsize=11)
        ax.set_ylabel("Density" if col_idx == 0 else "", fontsize=11)
        ax.set_title(f"N = {n_samples:,}", fontsize=12)
        ax.legend(fontsize=8)
    
    fig.suptitle(f"Bayes Factor Distribution ({sc_label}, sigma={config.SIGMA})",
                 fontsize=14, y=1.02)
    plt.tight_layout()
    fig_path = os.path.join(config.OUTPUT_DIR, f"bayes_factor_histogram_{sc}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Figure saved: {fig_path}")

# ============================================================================
# セル11: 可視化4 - 捕捉成功率の推移（線グラフ）
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for idx, sc in enumerate(["equal", "accel"]):
    ax = axes[idx]
    df_sc = results_df[results_df["scenario"] == sc]
    
    ax.plot(df_sc["N"], df_sc["capture_rate_chi2"], 'o-', color='#E74C3C',
            linewidth=2, markersize=8, label="Chi-square (frequentist)")
    ax.plot(df_sc["N"], df_sc["capture_rate_bayes"], 's-', color='#2E86C1',
            linewidth=2, markersize=8, label="Bayesian (Bayes Factor)")
    
    ax.axhline(y=95, color='gray', linestyle='--', alpha=0.5, label='95% target')
    ax.set_xlabel("Trial Count N", fontsize=13)
    ax.set_ylabel("Capture Success Rate (%)", fontsize=13)
    ax.set_xscale('log')
    
    title = "Equal-Interval" if sc == "equal" else "Accelerated Degradation"
    ax.set_title(f"Capture Rate vs N: {title}", fontsize=13)
    ax.set_ylim(0, 105)
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(config.OUTPUT_DIR, "capture_rate_trend.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

# ============================================================================
# セル12: 可視化5 - 健全度曲線のシミュレーション（環境・材質別）
# ============================================================================

def simulate_health_curve(p_ab, p_bc, years=40, c_km=1.0):
    """健全度 H の経年変化をシミュレート"""
    W = np.array([1.0, 0.5, 0.3, 0.1])  # 基準健全度 [A, B, C, D]
    P = np.array([1.0, 0.0, 0.0, 0.0])  # 初期状態（全てA）
    
    H_history = [np.dot(W, P)]
    
    for year in range(1, years + 1):
        p_ab_eff = min(p_ab * c_km, 0.95)
        p_bc_eff = min(p_bc * c_km, 0.95)
        p_cd = min(0.15 * c_km, 0.95)
        
        P_new = np.zeros(4)
        P_new[0] = P[0] * (1 - p_ab_eff)
        P_new[1] = P[0] * p_ab_eff + P[1] * (1 - p_bc_eff)
        P_new[2] = P[1] * p_bc_eff + P[2] * (1 - p_cd)
        P_new[3] = P[2] * p_cd + P[3]
        
        P = P_new
        H_history.append(np.dot(W, P))
    
    return np.array(H_history)


fig, ax = plt.subplots(figsize=(14, 8))

scenarios_ckm = {
    "Standard (C=1.0)": 1.0,
    "Salt Damage (C=2.2)": 2.2,
    "Salt + Coating (C=1.1)": 1.1,
    "SUS304 (C=0.7)": 0.7,
    "Zinc Plating (C=1.5)": 1.5,
}
colors_ckm = ['#2C3E50', '#E74C3C', '#F39C12', '#27AE60', '#8E44AD']

years = np.arange(0, 41)
for (label, c_km), color in zip(scenarios_ckm.items(), colors_ckm):
    H = simulate_health_curve(config.P_AB_TRUE, config.P_BC_TRUE, 40, c_km)
    ax.plot(years, H * 100, '-', color=color, linewidth=2.5, label=label)

ax.axhline(y=30, color='red', linestyle='--', linewidth=2, alpha=0.7,
           label='Renewal Threshold (H=30%)')
ax.set_xlabel("Years", fontsize=14)
ax.set_ylabel("Health Index H (%)", fontsize=14)
ax.set_title("Health Curve by Environment/Material Factor $C_{km}$\n"
             f"(P_AB={config.P_AB_TRUE}, P_BC={config.P_BC_TRUE})", fontsize=14)
ax.set_xlim(0, 40)
ax.set_ylim(0, 105)
ax.legend(fontsize=11, loc='upper right')
ax.grid(True, alpha=0.3)

# 9年と27年のマーカー
ax.axvline(x=9, color='gray', linestyle=':', alpha=0.5)
ax.axvline(x=27, color='gray', linestyle=':', alpha=0.5)
ax.text(9, 5, '9 yr\n(reactive)', ha='center', fontsize=10, color='gray')
ax.text(27, 5, '27 yr\n(preventive)', ha='center', fontsize=10, color='gray')

plt.tight_layout()
fig_path = os.path.join(config.OUTPUT_DIR, "health_curves_ckm.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

# ============================================================================
# セル13: 可視化6 - 遷移劣化度 I_km の経時推移
# ============================================================================

def compute_transition_intensity(p_ab, p_bc, years=40, c_km=1.0):
    """遷移劣化度 I_AB, I_BC の経年推移"""
    W = np.array([1.0, 0.5, 0.3, 0.1])
    P = np.array([1.0, 0.0, 0.0, 0.0])
    
    I_AB_list = []
    I_BC_list = []
    
    p_ab_eff = min(p_ab * c_km, 0.95)
    p_bc_eff = min(p_bc * c_km, 0.95)
    
    for year in range(years):
        # 遷移劣化度 = 遷移確率 × 状態確率 × 健全度差分
        I_AB = p_ab_eff * P[0] * (W[0] - W[1])  # A→Bの遷移劣化度
        I_BC = p_bc_eff * P[1] * (W[1] - W[2])  # B→Cの遷移劣化度
        
        I_AB_list.append(I_AB)
        I_BC_list.append(I_BC)
        
        p_cd = min(0.15 * c_km, 0.95)
        P_new = np.zeros(4)
        P_new[0] = P[0] * (1 - p_ab_eff)
        P_new[1] = P[0] * p_ab_eff + P[1] * (1 - p_bc_eff)
        P_new[2] = P[1] * p_bc_eff + P[2] * (1 - p_cd)
        P_new[3] = P[2] * p_cd + P[3]
        P = P_new
    
    return np.array(I_AB_list), np.array(I_BC_list)


fig, axes = plt.subplots(1, 2, figsize=(16, 6))
years_plot = np.arange(0, 40)

for (label, c_km), color in zip(scenarios_ckm.items(), colors_ckm):
    I_AB, I_BC = compute_transition_intensity(
        config.P_AB_TRUE, config.P_BC_TRUE, 40, c_km
    )
    axes[0].plot(years_plot, I_AB, '-', color=color, linewidth=2, label=label)
    axes[1].plot(years_plot, I_BC, '-', color=color, linewidth=2, label=label)

for ax, title in zip(axes, ["I_AB (A->B)", "I_BC (B->C)"]):
    ax.set_xlabel("Years", fontsize=13)
    ax.set_ylabel("Transition Intensity", fontsize=13)
    ax.set_title(f"Transition Intensity {title}", fontsize=13)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(config.OUTPUT_DIR, "transition_intensity.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

# ============================================================================
# セル14: 可視化7 - 跳躍遷移 A→C の発生確率分析
# ============================================================================

def compute_jump_probability(n_units, n_steps, p_ab_annual, p_bc_annual,
                              scenario, n_trials=10000):
    """A→C跳躍遷移の発生確率を計算"""
    rng = np.random.default_rng(123)
    p_ab_step, p_bc_step = compute_micro_step_probabilities(
        p_ab_annual, p_bc_annual, n_steps
    )
    
    jump_count = 0
    for _ in range(n_trials):
        if scenario == "equal":
            final = simulate_single_unit_equal(p_ab_step, p_bc_step, n_steps, rng)
        else:
            final = simulate_single_unit_accel(p_ab_step, p_bc_step, n_steps, rng)
        
        # 初期状態Aから直接Cに到達 = 跳躍遷移
        if final == 2:
            jump_count += 1
    
    return jump_count / n_trials

# ステップ数を変えて跳躍遷移確率を計算
step_counts = [1, 2, 4, 6, 12, 24, 52, 100]
jump_probs_equal = []
jump_probs_accel = []

for ns in step_counts:
    jp_eq = compute_jump_probability(
        1000, ns, config.P_AB_TRUE, config.P_BC_TRUE, "equal"
    )
    jp_ac = compute_jump_probability(
        1000, ns, config.P_AB_TRUE, config.P_BC_TRUE, "accel"
    )
    jump_probs_equal.append(jp_eq)
    jump_probs_accel.append(jp_ac)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(step_counts, [p*100 for p in jump_probs_equal], 'o-', color='#2E86C1',
        linewidth=2, markersize=8, label='Equal-Interval')
ax.plot(step_counts, [p*100 for p in jump_probs_accel], 's-', color='#E74C3C',
        linewidth=2, markersize=8, label='Accelerated')

ax.set_xlabel("Micro-steps per Period (n)", fontsize=13)
ax.set_ylabel("Jump Transition A->C Probability (%)", fontsize=13)
ax.set_title("Jump Transition Probability vs Micro-step Resolution", fontsize=14)
ax.set_xscale('log')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(config.OUTPUT_DIR, "jump_transition_probability.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

# ============================================================================
# セル15: 結果の評価と統計的検証
# ============================================================================

print("\n" + "=" * 70)
print("【結果の評価と統計的検証】")
print("=" * 70)

for sc in ["equal", "accel"]:
    sc_label = "等間隔遷移" if sc == "equal" else "加速劣化"
    print(f"\n--- {sc_label}シナリオ ---")
    df_sc = results_df[results_df["scenario"] == sc]
    
    for _, row in df_sc.iterrows():
        n = int(row["N"])
        chi2_rate = row["capture_rate_chi2"]
        bayes_rate = row["capture_rate_bayes"]
        p_mean = row["p_value_mean"]
        p_median = row["p_value_median"]
        
        print(f"\n  N = {n:>6,}:")
        print(f"    Chi-square  : capture rate = {chi2_rate:5.1f}%  "
              f"(p-mean={p_mean:.4f}, p-median={p_median:.4f})")
        print(f"    Bayesian    : capture rate = {bayes_rate:5.1f}%  "
              f"(log10(BF)-mean={row['log_bf_mean']:.2f})")
        
        if chi2_rate < 50 and bayes_rate > 90:
            print(f"    >> Chi-square overfitting detected. Bayesian approach stable.")
        elif bayes_rate > 95:
            print(f"    >> Bayesian capture highly robust at this N.")

# 健全度30%到達年の計算
print("\n\n" + "=" * 70)
print("【健全度30%到達年の予測（環境・材質別）】")
print("=" * 70)

for label, c_km in scenarios_ckm.items():
    H = simulate_health_curve(config.P_AB_TRUE, config.P_BC_TRUE, 60, c_km)
    reach_year = None
    for y, h in enumerate(H):
        if h * 100 <= 30:
            reach_year = y
            break
    if reach_year:
        print(f"  {label:30s}: H=30% at year {reach_year}")
    else:
        print(f"  {label:30s}: H=30% not reached within 60 years")

# ============================================================================
# セル16: 追加検証 - σ感度分析
# ============================================================================

print("\n" + "=" * 70)
print("【追加検証: 評価ノイズσの感度分析】")
print("=" * 70)

sigma_values = [0.01, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20]
n_fixed = 5000
sensitivity_results = []

theo_dist_eq = theo_dists["equal"]
rng_sens = np.random.default_rng(99)

for sigma_test in sigma_values:
    success_chi2 = 0
    success_bayes = 0
    
    for _ in range(config.N_REPEATS):
        obs, _ = simulate_population(
            n_fixed, "equal", config.P_AB_TRUE, config.P_BC_TRUE,
            config.N_STEPS, sigma_test, rng_sens
        )
        
        p_val, _ = chi_square_test(obs, theo_dist_eq)
        if p_val > 0.05:
            success_chi2 += 1
        
        bf, _ = bayes_factor_test(obs, theo_dist_eq, config.KAPPA_PRIOR, sigma_test)
        if bf > 1/3:
            success_bayes += 1
    
    rate_chi2 = success_chi2 / config.N_REPEATS * 100
    rate_bayes = success_bayes / config.N_REPEATS * 100
    
    sensitivity_results.append({
        "sigma": sigma_test,
        "capture_rate_chi2": rate_chi2,
        "capture_rate_bayes": rate_bayes
    })
    print(f"  sigma={sigma_test:.2f}: Chi2={rate_chi2:.1f}%, Bayes={rate_bayes:.1f}%")

sens_df = pd.DataFrame(sensitivity_results)
csv_path_sens = os.path.join(config.OUTPUT_DIR, "sigma_sensitivity.csv")
sens_df.to_csv(csv_path_sens, index=False, encoding='utf-8-sig')

# σ感度グラフ
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sens_df["sigma"], sens_df["capture_rate_chi2"], 'o-', color='#E74C3C',
        linewidth=2, markersize=8, label="Chi-square")
ax.plot(sens_df["sigma"], sens_df["capture_rate_bayes"], 's-', color='#2E86C1',
        linewidth=2, markersize=8, label="Bayesian")

ax.set_xlabel("Evaluation Noise sigma", fontsize=13)
ax.set_ylabel("Capture Success Rate (%)", fontsize=13)
ax.set_title(f"Sensitivity to Evaluation Noise (N={n_fixed:,}, Equal-Interval)",
             fontsize=14)
ax.axhline(y=95, color='gray', linestyle='--', alpha=0.5)
ax.set_ylim(0, 105)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

plt.tight_layout()
fig_path = os.path.join(config.OUTPUT_DIR, "sigma_sensitivity.png")
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved: {fig_path}")

# ============================================================================
# セル17: 最終サマリーと出力ファイル一覧
# ============================================================================

print("\n" + "=" * 70)
print("【最終サマリー】")
print("=" * 70)
print(f"""
本シミュレーションの主要知見:

1. 頻度主義的アプローチ（カイ二乗検定）の限界:
   - N の増大に伴い、評価ブレ(sigma={config.SIGMA})を「統計的に有意な差」として
     過剰検出し、同一集団の捕捉成功率が低下する傾向を確認。
   - 特に N >= 5,000 以上で捕捉率の低下が顕著。

2. ベイズ的アプローチの安定性:
   - 事前分布に評価ブレの知見を組み込むことで、N に依存しない
     ロバストな同一集団捕捉（捕捉率 95% 以上）を実現。
   - 加速劣化シナリオにおいても安定した捕捉を維持。

3. 健全度30%到達年の延命効果:
   - 標準条件(C=1.0)での予防保全により、事後保全（約9年）を
     大幅に延命可能であることを定量的に確認。
   - 塩害地域でも塗装介入により延命効果を維持。

4. 跳躍遷移 A→C の確率的分解:
   - 微細ステップの増加により、跳躍遷移を連続事象の積として
     評価可能であることを確認。
""")

print("\n【出力ファイル一覧】")
for f in sorted(os.listdir(config.OUTPUT_DIR)):
    fpath = os.path.join(config.OUTPUT_DIR, f)
    size = os.path.getsize(fpath)
    print(f"  {f:45s} ({size:>10,} bytes)")

print("\n完了。")
